In [6]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
import os
import requests
from dotenv import load_dotenv

In [ ]:
@tool
def get_ex_rate(base_currency: str, target_currency: str) -> str:
    """returns exchange rate for currency"""
    
    url = f"https://api.frankfurter.dev/v2/rate/{base_currency}/{target_currency}"
    response = requests.get(url)
    data = response.json()
    return f'1 {data['base']} = {data['rate']} {data['quote']}'


In [18]:
load_dotenv()
GEMINI_API_KEY=os.getenv('GEMINI_API_KEY')

In [32]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=.7,
    api_version='v1beta'
)
agent = create_agent(
    model=model,
    tools=[get_ex_rate],
    system_prompt='you are helpful assistant that uses tools when needed',
    
)

In [33]:
config = {'configurable': {'thread_id': 'session-1'}}

base = input('What currency do you want to know the exchange rate for? ').lower()
pop_currency = ['npr', 'inr', 'eur', 'gbp', 'usd', 'cad']

# Join the list into a comma-separated string
currencies_str = ", ".join(pop_currency)
ex_rate = f'What are the exchange rates from {base} to all of these currencies: {currencies_str}?'

r = agent.invoke(
    {'messages': [{'role': 'user', 'content': ex_rate}]},
    config,
)
print(r['messages'][-1].content)

[{'type': 'text', 'text': 'Here are the current exchange rates from 1 Nepalese Rupee (NPR) to the requested currencies:\n\n*   **NPR:** 1.0000\n*   **INR:** 0.6226\n*   **EUR:** 0.0060\n*   **GBP:** 0.0050\n*   **USD:** 0.0066\n*   **CAD:** 0.0094', 'extras': {'signature': 'EnMKcQFpFH0Ty6xJYlCUq8KMVx3v6JrMqXpoHUDL/nYMzYferln7aJ02NNYUy554VHTytgSp+jC7Yp1eBtrM8UyAakE2xPFo6IPh7uakHJ5MOaSn8qwEbbNa9G/O9u6DaqkG8CE8tWaDek/DZoV0N6qkiWA8'}}]
